# 🎬 Wan 2.2 Remix I2V - Modal Deployment

One-click deployment of Wan 2.2 Remix on Modal A100 GPUs.

## Instructions
1. Enter your Civitai API key in Cell 2
2. Run all cells in order (Shift+Enter)
3. Open the ComfyUI URL displayed after Cell 4
4. Load your workflow JSON and start generating

In [ ]:
# @title 🔧 Environment Setup
import modal
import os

app = modal.App("wan-remix-i2v")

cuda_version = "12.4.0"
os_version = "ubuntu22.04"
tag = f"{cuda_version}-devel-{os_version}"

image = (
    modal.Image.from_registry(f"nvidia/cuda:{tag}", add_python="3.11")
    .apt_install("git", "wget", "curl", "libgl1-mesa-glx", "libglib2.0-0", "libgomp1")
    .pip_install("comfy-cli", "huggingface_hub")
    .run_commands("comfy --workspace /root/comfyui install --nvidia")
)

volume = modal.Volume.from_name("wan-remix-models", create_if_missing=True)
print("✅ Environment ready!")

In [ ]:
# @title 📦 Configuration - PASTE YOUR CIVITAI API KEY BELOW
import os

# 🔑 REPLACE THIS WITH YOUR ACTUAL CIVITAI API KEY
CIVITAI_API_KEY = "eafdf0d7ecc4b5e39e2739bdf1d9a5e1"
os.environ["CIVITAI_API_KEY"] = CIVITAI_API_KEY

CIVITAI_MODEL_ID = "2770795"
CIVITAI_URL = f"https://civitai.com/api/download/models/{CIVITAI_MODEL_ID}?type=Model&format=SafeTensor&size=pruned&fp=fp8&token={CIVITAI_API_KEY}"

MODELS = {
    "unet": {
        "url": CIVITAI_URL,
        "filename": "Wan2.2_Remix_NSFW_i2v_14b_high_lighting_fp8_e4m3fn_v3.0.safetensors"
    },
    "vae": {
        "url": "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors",
        "filename": "wan_2.1_vae.safetensors"
    },
    "clip": {
        "url": "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/nsfw_wan_umt5-xxl_fp8_scaled.safetensors",
        "filename": "nsfw_wan_umt5-xxl_fp8_scaled.safetensors"
    }
}

CUSTOM_NODES = [
    "https://github.com/rgthree/rgthree-comfy",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/cubiq/ComfyUI_essentials",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts",
    "https://github.com/if-ai/ComfyUI-MixLab",
    "https://github.com/chrisgoringego/cg-nodes",
]

print("✅ Configuration ready!")
print(f"📥 Will download {len(MODELS)} models + {len(CUSTOM_NODES)} custom nodes")

In [ ]:
# @title ⬇️ Download All Models & Custom Nodes
import subprocess
import pathlib

@app.function(
    image=image,
    volumes={"/root/comfyui": volume},
    gpu="A100",
    timeout=3600,
)
def download_models():
    import os
    workspace = pathlib.Path("/root/comfyui")
    
    for model_type, info in MODELS.items():
        target_dir = workspace / "models" / model_type
        target_dir.mkdir(parents=True, exist_ok=True)
        model_path = target_dir / info["filename"]
        if not model_path.exists():
            print(f"⬇️  Downloading {info['filename']}...")
            cmd = f'curl -L -C - -o "{model_path}" "{info["url"]}"'
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            if result.returncode != 0:
                print(f"❌ Failed: {result.stderr}")
            else:
                print("✅ Downloaded!")
        else:
            print(f"✅ Already exists: {info['filename']}")
    
    unet_dir = workspace / "models" / "unet"
    original = unet_dir / "Wan2.2_Remix_NSFW_i2v_14b_high_lighting_fp8_e4m3fn_v3.0.safetensors"
    symlink = unet_dir / "Wan2.2_Remix_NSFW_i2v_14b_low_lighting_fp8_e4m3fn_v3.0.safetensors"
    if not symlink.exists():
        os.symlink(original, symlink)
        print("🔗 Symlink created")
    
    nodes_dir = workspace / "custom_nodes"
    nodes_dir.mkdir(parents=True, exist_ok=True)
    for node_url in CUSTOM_NODES:
        repo_name = node_url.rstrip("/").split("/")[-1].replace(".git", "")
        node_path = nodes_dir / repo_name
        if not node_path.exists():
            print(f"⬇️  Cloning {repo_name}...")
            subprocess.run(["git", "clone", node_url, str(node_path)], check=True)
            req_file = node_path / "requirements.txt"
            if req_file.exists():
                subprocess.run(["pip", "install", "-r", str(req_file)], check=True)
            print("✅ Installed!")
        else:
            print(f"✅ Already exists: {repo_name}")
    print("\n🎉 All models and custom nodes are ready!")

with modal.enable_output():
    download_models.remote()

In [ ]:
# @title 🚀 Launch ComfyUI
import subprocess

@app.function(
    image=image,
    volumes={"/root/comfyui": volume},
    gpu="A100",
    timeout=86400,
)
@modal.web_server(port=8188)
def comfyui_server():
    download_models.remote()
    print("🚀 Starting ComfyUI...")
    subprocess.run(
        "comfy --workspace /root/comfyui launch -- --listen 0.0.0.0 --port 8188",
        shell=True, check=True
    )

with modal.enable_output():
    comfyui_server.remote()

## 🎉 Your ComfyUI is running!

### 📋 Next Steps:
1. Open the URL above in your browser
2. Click **Load** and select your Wan2.2 Remix workflow JSON
3. Upload a start image and type your prompt
4. Click **Queue Prompt**

### 🔗 Resources:
- [Wan2.2 Remix on Civitai](https://civitai.com/models/2003153/wan22-remix-t2vandi2v)
- [Workflow by 肥猴](https://www.runninghub.ai/workflow/2051406785923305474)